# Phase 1 — Set up both organisms

See `docs/research_proposal.md` §4.1-4.2. **Organism A** (multimodal induction: real image + `harmful_response`) reuses the released checkpoint when possible. **Organism B** (text-only induction: `image_desc` text + `harmful_response`, same underlying rows, no image ever shown) always fine-tunes -- no released text-only variant exists. Holding harm content fixed and varying only induction modality is the point (proposal §1).

In [ ]:
%pip install -q unsloth peft transformers trl datasets huggingface_hub accelerate bitsandbytes pillow pyyaml anthropic
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/emergent-misalignment-project'  # upload src/ here
sys.path.append(PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)  # src/ modules use paths relative to the project root (e.g. data/eval/scenarios.json)
ARTIFACTS = f'{PROJECT_DIR}/artifacts'


## Organism A — multimodal induction

### Step 0 — check for the released checkpoint/dataset first

In [ ]:
from src.data import check_released_checkpoint, check_released_dataset

checkpoint_a = check_released_checkpoint(local_dir=f'{ARTIFACTS}/checkpoints')
dataset_path = check_released_dataset()  # informational only now -- see below
checkpoint_a, dataset_path


### Steps 1-2 — build data + fine-tune, only if no released checkpoint was found

The released checkpoint (r=128) covers Organism A entirely -- this whole block is skipped when `checkpoint_a` is already set above. Kept as a fallback (e.g. if the release ever disappears), using the released dataset in preference to synthesizing one, and only falling back to `harm_prompt_fn` if that's unavailable too.

In [ ]:
if checkpoint_a is None:
    from src.data import build_multimodal_harm_dataset
    from src.train import load_base_model, train_lora

    if dataset_path is None:
        from src.harm_generation import make_llm_harm_prompt_fn, openai_compatible_llm_call  # or anthropic_llm_call

        IMAGE_DATASET = ''  # e.g. a public face/photo dataset repo id on the HF Hub — fill in and verify

        # See src/harm_generation.py's module docstring before choosing a provider — commercial
        # chat APIs are likely to refuse/filter this content at scale; the original paper used a
        # self-hosted open-weight model (Qwen3-235B via vLLM), not a commercial API.
        harm_prompt_fn = make_llm_harm_prompt_fn(
            lambda system, image: openai_compatible_llm_call(
                system, image, model='', base_url='',  # fill in your self-hosted/OpenAI-compatible endpoint
            )
        )
        dataset_path = build_multimodal_harm_dataset(
            harm_prompt_fn, image_dataset=IMAGE_DATASET, n_examples=1200,
            out_dir=f'{ARTIFACTS}/data/multimodal_harm',
        )

    model_a, tokenizer_a = load_base_model()
    checkpoint_a = train_lora(model_a, tokenizer_a, dataset_path, output_dir=f'{ARTIFACTS}/checkpoints/multimodal_em')

checkpoint_a


In [ ]:
from pathlib import Path

Path(f'{ARTIFACTS}/checkpoint_path_A.txt').write_text(checkpoint_a)


## Organism B — text-only induction (new)

Same underlying rows as Organism A, same LoRA recipe, but `image_desc` (text) stands in for the image and no image is ever shown during training — isolates induction modality as the sole variable (proposal §4.2). No released checkpoint to check for; always trains.

In [ ]:
from src.data import build_text_only_organism_dataset
from src.train import load_base_model, train_lora

dataset_path_b = build_text_only_organism_dataset(out_dir=f'{ARTIFACTS}/data/text_only_organism')
model_b, tokenizer_b = load_base_model()
checkpoint_b = train_lora(model_b, tokenizer_b, dataset_path_b, output_dir=f'{ARTIFACTS}/checkpoints/text_only_em')
checkpoint_b


In [ ]:
Path(f'{ARTIFACTS}/checkpoint_path_B.txt').write_text(checkpoint_b)
